In [1]:
print("hello")

hello


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_FOLDER = Path(
    r"C:\Users\pkmpp\Documents\GitHub\J26-DS-323\experiments\pasan\step by step\data"
)

# Change this path if your five-minute file is elsewhere.
INPUT_FILE = BASE_FOLDER / "solar_5min_with_time_features_03 combine_wether_data.csv"

OUTPUT_FOLDER = BASE_FOLDER / "device_lifetime_analysis_5min_04 check each devices"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000

# Assumes pac is already in watts.
PAC_TO_WATTS = 1.0

# Common comparison window: 06:00 inclusive to 19:00 exclusive.
EXPECTED_READINGS_PER_DAY = 13 * 60 // 5  # 156

# Used for the highlighted daily means and monthly summaries.
MIN_DAILY_COVERAGE = 0.90

required = [
    "device_number",
    "inverterSn",
    "file",
    "timestamp_5min",
    "pac",
    "pv_capacity_kwp",
]

pieces = []

with pd.read_csv(
    INPUT_FILE,
    usecols=required,
    dtype={
        "device_number": "string",
        "inverterSn": "string",
    },
    encoding="utf-8-sig",
    chunksize=CHUNK_SIZE,
) as reader:

    for chunk in reader:
        chunk["_time"] = pd.to_datetime(
            chunk["timestamp_5min"],
            utc=True,
            format="mixed",
            errors="raise",
        ).dt.tz_convert("Asia/Colombo")

        chunk["pac"] = pd.to_numeric(
            chunk["pac"], errors="raise"
        )

        chunk["pv_capacity_kwp"] = pd.to_numeric(
            chunk["pv_capacity_kwp"], errors="raise"
        )

        pieces.append(chunk)

df = pd.concat(pieces, ignore_index=True)

if df.empty:
    raise ValueError("No data rows found")

if (
    df["device_number"].isna().any()
    or df["inverterSn"].isna().any()
    or df["_time"].isna().any()
):
    raise ValueError("Missing device identifiers or timestamps")

if df.duplicated(["device_number", "_time"]).any():
    raise ValueError(
        "Duplicate device/timestamp pairs found. Resolve these first."
    )

if not df["_time"].eq(df["_time"].dt.floor("5min")).all():
    raise ValueError("Some timestamps are not on the five-minute grid")

# Ensure a device number does not combine different inverters.
serial_counts = df.groupby("device_number")["inverterSn"].nunique()

if serial_counts.gt(1).any():
    raise ValueError(
        "These device numbers map to multiple inverter serials: "
        f"{serial_counts[serial_counts.gt(1)].index.tolist()}"
    )

minutes = df["_time"].dt.hour * 60 + df["_time"].dt.minute

df = df.loc[
    minutes.ge(6 * 60)
    & minutes.lt(19 * 60)
].copy()

if df.empty:
    raise ValueError("No readings within the selected time window")

valid_power = np.isfinite(df["pac"]) & df["pac"].ge(0)

print("Invalid power readings excluded:", int((~valid_power).sum()))

# Retain invalid timestamps as observations, but omit their power
# from statistics and valid-reading coverage.
df["_power_kw"] = (
    df["pac"] * PAC_TO_WATTS / 1000
).where(valid_power)

df["_date"] = df["_time"].dt.date

df = df.sort_values(
    ["device_number", "_time"]
).reset_index(drop=True)

print("Rows:", f"{len(df):,}")
print("Devices:", df["device_number"].nunique())
print("Dates:", df["_time"].min(), "to", df["_time"].max())

Invalid power readings excluded: 0
Rows: 2,767,521
Devices: 49
Dates: 2023-06-01 16:35:00+05:30 to 2026-08-01 18:55:00+05:30


## Calculate daily and monthly summaries

In [3]:
from IPython.display import display

In [4]:
daily_parts = []
monthly_parts = []

for device_number, group in df.groupby(
    "device_number", sort=True
):
    daily = group.groupby("_date").agg(
        recorded_readings=("_time", "size"),
        valid_readings=("_power_kw", "count"),
        daily_mean_kw=("_power_kw", "mean"),
        daily_p95_kw=(
            "_power_kw",
            lambda values: values.quantile(0.95),
        ),
        daily_peak_kw=("_power_kw", "max"),
    )

    # Include absent days within this device's observed history.
    calendar = pd.date_range(
        min(group["_date"]),
        max(group["_date"]),
        freq="D",
    )

    daily.index = pd.to_datetime(daily.index)
    daily = daily.reindex(calendar)
    daily.index.name = "date"

    for column in ["recorded_readings", "valid_readings"]:
        daily[column] = (
            daily[column].fillna(0).astype(int)
        )

    daily["coverage_fraction"] = (
        daily["valid_readings"] / EXPECTED_READINGS_PER_DAY
    )

    daily["coverage_ok"] = daily["coverage_fraction"].ge(
        MIN_DAILY_COVERAGE
    )

    daily["qualified_daily_mean_kw"] = (
        daily["daily_mean_kw"].where(daily["coverage_ok"])
    )

    daily["qualified_daily_p95_kw"] = (
        daily["daily_p95_kw"].where(daily["coverage_ok"])
    )

    # Do not draw a rolling estimate through a long data gap.
    rolling_count = (
        daily["qualified_daily_mean_kw"]
        .rolling("30D")
        .count()
    )

    daily["rolling_30day_mean_kw"] = (
        daily["qualified_daily_mean_kw"]
        .rolling("30D")
        .mean()
        .where(rolling_count.ge(10) & daily["coverage_ok"])
    )

    daily["device_number"] = device_number

    monthly = daily.resample("MS").agg(
        observed_calendar_days=("coverage_fraction", "size"),
        days_with_readings=(
            "valid_readings",
            lambda values: values.gt(0).sum(),
        ),
        qualified_days=("coverage_ok", "sum"),
        monthly_mean_kw=("qualified_daily_mean_kw", "mean"),
        mean_daily_p95_kw=("qualified_daily_p95_kw", "mean"),
        average_daily_coverage=("coverage_fraction", "mean"),
    )

    monthly["device_number"] = device_number

    daily_parts.append(daily.reset_index())
    monthly_parts.append(
        monthly.rename_axis("month").reset_index()
    )

daily_summary = pd.concat(daily_parts, ignore_index=True)
monthly_summary = pd.concat(monthly_parts, ignore_index=True)

daily_summary.to_csv(
    OUTPUT_FOLDER / "daily_power_summary.csv",
    index=False,
)

monthly_summary.to_csv(
    OUTPUT_FOLDER / "monthly_power_summary.csv",
    index=False,
)

device_mapping = (
    df.groupby("device_number")
    .agg(
        inverterSn=("inverterSn", "first"),
        source_files=(
            "file",
            lambda values: " | ".join(sorted(values.dropna().unique())),
        ),
        first_timestamp=("_time", "min"),
        last_timestamp=("_time", "max"),
        capacity_values_kwp=(
            "pv_capacity_kwp",
            lambda values: " | ".join(
                map(str, sorted(values.dropna().unique()))
            ),
        ),
    )
    .reset_index()
)

device_mapping.to_csv(
    OUTPUT_FOLDER / "device_mapping.csv",
    index=False,
)

display(monthly_summary.head(10))

,month,observed_calendar_days,days_with_readings,qualified_days,monthly_mean_kw,mean_daily_p95_kw,average_daily_coverage,device_number
0,2024-08-01,5,5,2,1.368164,4.300250,0.532051,1
1,2024-10-01,1,1,0,NaN,NaN,0.403846,10
2,2024-11-01,30,29,21,1.017166,2.380024,0.860470,10
3,2024-12-01,31,30,27,1.284162,2.739593,0.855873,10
4,2025-01-01,31,13,9,1.087823,2.502278,0.354012,10
5,2025-02-01,28,1,0,NaN,NaN,0.004121,10
6,2025-03-01,31,7,3,1.261440,2.558167,0.187965,10
7,2025-04-01,30,19,16,1.154029,2.414656,0.580556,10
8,2025-05-01,31,0,0,NaN,NaN,0.000000,10
9,2025-06-01,30,10,7,1.020347,2.204571,0.257906,10


## Save one lifetime plot per device

In [5]:
for device_number, daily in daily_summary.groupby(
    "device_number", sort=True
):
    monthly = monthly_summary.loc[
        monthly_summary["device_number"].eq(device_number)
    ].copy()

    mapping = device_mapping.loc[
        device_mapping["device_number"].eq(device_number)
    ].iloc[0]

    daily = daily.sort_values("date")
    monthly = monthly.sort_values("month")

    fig, axes = plt.subplots(
        3, 1,
        figsize=(15, 10),
        sharex=True,
        gridspec_kw={"height_ratios": [2, 2, 1]},
    )

    # Daily values as points so missing days are not bridged.
    axes[0].scatter(
        daily["date"],
        daily["daily_mean_kw"],
        s=5,
        color="lightgray",
        label="Daily mean: all recorded days",
    )

    axes[0].scatter(
        daily["date"],
        daily["qualified_daily_mean_kw"],
        s=7,
        color="steelblue",
        label=f"Daily mean: coverage ≥ {MIN_DAILY_COVERAGE:.0%}",
    )

    axes[0].plot(
        daily["date"],
        daily["rolling_30day_mean_kw"],
        color="darkorange",
        linewidth=2,
        label="30-day rolling mean of qualified days",
    )

    axes[0].set_ylabel("Mean power (kW)")
    axes[0].legend(loc="best", fontsize=8)
    axes[0].grid(alpha=0.2)

    axes[1].plot(
        monthly["month"],
        monthly["monthly_mean_kw"],
        marker="o",
        label="Monthly mean of qualified daily means",
    )

    axes[1].plot(
        monthly["month"],
        monthly["mean_daily_p95_kw"],
        marker="s",
        label="Monthly mean of qualified daily P95 power",
    )

    axes[1].set_ylabel("Power (kW)")
    axes[1].legend(loc="best", fontsize=8)
    axes[1].grid(alpha=0.2)

    axes[2].bar(
        daily["date"],
        daily["coverage_fraction"] * 100,
        width=1,
        color="seagreen",
    )

    axes[2].axhline(
        MIN_DAILY_COVERAGE * 100,
        color="red",
        linestyle="--",
        linewidth=1,
    )

    axes[2].set_ylabel("Coverage (%)")
    axes[2].set_ylim(0, 105)
    axes[2].set_xlabel("Date")
    axes[2].grid(alpha=0.2)

    fig.suptitle(
        f"Device {device_number} | Inverter {mapping['inverterSn']}\n"
        f"Listed capacity: {mapping['capacity_values_kwp']} kWp | "
        "06:00–19:00 local time",
        fontsize=12,
    )

    fig.autofmt_xdate()
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    plot_file = (
        OUTPUT_FOLDER / f"device_{device_number}_lifetime.png"
    )

    plt.savefig(plot_file, dpi=180, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", plot_file.name)

print("\nAll reports and plots:", OUTPUT_FOLDER)

Saved: device_1_lifetime.png
Saved: device_10_lifetime.png
Saved: device_11_lifetime.png
Saved: device_12_lifetime.png
Saved: device_13_lifetime.png
Saved: device_14_lifetime.png
Saved: device_15_lifetime.png
Saved: device_16_lifetime.png
Saved: device_17_lifetime.png
Saved: device_18_lifetime.png
Saved: device_19_lifetime.png
Saved: device_2_lifetime.png
Saved: device_20_lifetime.png
Saved: device_21_lifetime.png
Saved: device_22_lifetime.png
Saved: device_23_lifetime.png
Saved: device_24_lifetime.png
Saved: device_25_lifetime.png
Saved: device_26_lifetime.png
Saved: device_27_lifetime.png
Saved: device_28_lifetime.png
Saved: device_29_lifetime.png
Saved: device_3_lifetime.png
Saved: device_30_lifetime.png
Saved: device_31_lifetime.png
Saved: device_32_lifetime.png
Saved: device_33_lifetime.png
Saved: device_34_lifetime.png
Saved: device_35_lifetime.png
Saved: device_36_lifetime.png
Saved: device_37_lifetime.png
Saved: device_38_lifetime.png
Saved: device_39_lifetime.png
Saved: device